# 04 — Projections: From RDF to Visualization

RDF is verbose by design — every fact is a triple, every identity is a URI,
blank nodes encode structure, reification annotates assertions.  This is
the right representation for semantic reasoning and SHACL validation.
It is the wrong representation for a force-directed graph layout.

Projections bridge that gap.  This notebook demonstrates:

1. **Type collapse** — `rdf:type` becomes a node attribute, not an edge
2. **Literal collapse** — data properties become node attributes
3. **Blank node resolution** — inline structured values as nested dicts
4. **RDF list resolution** — `rdf:first`/`rdf:rest` chains become Python lists
5. **Reification collapse** — `rdf:Statement` becomes an edge with metadata
6. **CONSTRUCT-based projections** — graph-to-graph transforms that stay in RDF
7. **Projection pipelines** — composable chains of CONSTRUCT and Python steps
8. **Holarchy-level projections** — project holons and the holarchy topology

Two modes: **Graph-to-Graph** (CONSTRUCT, stays in RDF) and
**Graph-to-Structure** (Pythonic, exits RDF into dicts/LPG).

In [ ]:
from rdflib import Graph, Literal, Namespace, URIRef, BNode
from rdflib.namespace import RDF, RDFS, XSD, SKOS
from rdflib.collection import Collection

from holonic import HolonicDataset
from holonic.projections import (
    project_to_lpg,
    collapse_reification,
    extract_types,
    filter_by_class,
    localize_predicates,
    strip_blank_nodes,
    build_construct,
    ProjectionPipeline,
    CONSTRUCT_STRIP_TYPES,
    CONSTRUCT_OBJECT_PROPERTIES_ONLY,
    CONSTRUCT_DATA_PROPERTIES_ONLY,
    CONSTRUCT_LABELS_ONLY,
    CONSTRUCT_SUBCLASS_TREE,
    CONSTRUCT_COLLAPSE_REIFICATION,
)

SEP = "=" * 65

EX = Namespace("urn:ex:")
SENSOR = Namespace("urn:sensor:")
MSN = Namespace("urn:mission:")

---
## 1. Build a Rich RDF Graph

This graph has everything: typed nodes, literal properties, blank nodes
for structured values, an RDF collection (list), and reified statements.
It represents a small air operations scenario — the kind of data that
would live inside holons in a defense interop system.

In [ ]:
g = Graph()
g.bind("ex", EX)
g.bind("sensor", SENSOR)
g.bind("msn", MSN)

# ── Aircraft with properties ──
g.add((EX.aircraft_001, RDF.type, EX.Aircraft))
g.add((EX.aircraft_001, RDFS.label, Literal("Viper-01")))
g.add((EX.aircraft_001, EX.callsign, Literal("VIPER01")))
g.add((EX.aircraft_001, EX.latitude, Literal(34.05, datatype=XSD.double)))
g.add((EX.aircraft_001, EX.longitude, Literal(-118.25, datatype=XSD.double)))
g.add((EX.aircraft_001, EX.altitude_m, Literal(10000, datatype=XSD.integer)))
g.add((EX.aircraft_001, EX.speed_kts, Literal(450, datatype=XSD.integer)))
g.add((EX.aircraft_001, EX.heading_deg, Literal(270.0, datatype=XSD.double)))
g.add((EX.aircraft_001, EX.assigned_to, EX.mission_alpha))

# ── Blank node: sensor suite (structured value) ──
sensor_suite = BNode()
g.add((EX.aircraft_001, EX.sensorSuite, sensor_suite))
g.add((sensor_suite, SENSOR.radar, Literal("AN/APG-83")))
g.add((sensor_suite, SENSOR.eo_ir, Literal("AN/AAQ-33")))
g.add((sensor_suite, SENSOR.ew, Literal("AN/ALQ-250")))

# ── RDF list: loadout ──
loadout_items = [Literal("AIM-120D"), Literal("AIM-9X"), Literal("GBU-31")]
loadout_head = BNode()
Collection(g, loadout_head, loadout_items)
g.add((EX.aircraft_001, EX.loadout, loadout_head))

# ── Second aircraft ──
g.add((EX.aircraft_002, RDF.type, EX.Aircraft))
g.add((EX.aircraft_002, RDFS.label, Literal("Reaper-03")))
g.add((EX.aircraft_002, EX.callsign, Literal("REAPER03")))
g.add((EX.aircraft_002, EX.latitude, Literal(34.12, datatype=XSD.double)))
g.add((EX.aircraft_002, EX.longitude, Literal(-118.19, datatype=XSD.double)))
g.add((EX.aircraft_002, EX.altitude_m, Literal(8500, datatype=XSD.integer)))
g.add((EX.aircraft_002, EX.assigned_to, EX.mission_bravo))

# ── Mission nodes ──
g.add((EX.mission_alpha, RDF.type, MSN.Mission))
g.add((EX.mission_alpha, RDFS.label, Literal("Mission Alpha")))
g.add((EX.mission_alpha, MSN.missionType, Literal("CAS")))
g.add((EX.mission_alpha, MSN.priority, Literal(1, datatype=XSD.integer)))

g.add((EX.mission_bravo, RDF.type, MSN.Mission))
g.add((EX.mission_bravo, RDFS.label, Literal("Mission Bravo")))
g.add((EX.mission_bravo, MSN.missionType, Literal("ISR")))
g.add((EX.mission_bravo, MSN.priority, Literal(2, datatype=XSD.integer)))

# ── Target ──
g.add((EX.target_001, RDF.type, EX.Target))
g.add((EX.target_001, RDFS.label, Literal("TGT-ALPHA-7")))
g.add((EX.target_001, EX.latitude, Literal(34.08, datatype=XSD.double)))
g.add((EX.target_001, EX.longitude, Literal(-118.22, datatype=XSD.double)))
g.add((EX.mission_alpha, MSN.hasTarget, EX.target_001))

# ── Reified statement: confidence-annotated identification ──
stmt = EX.identification_stmt
g.add((stmt, RDF.type, RDF.Statement))
g.add((stmt, RDF.subject, EX.aircraft_001))
g.add((stmt, RDF.predicate, EX.identifiedAs))
g.add((stmt, RDF.object, Literal("F-16C Block 70")))
g.add((stmt, EX.confidence, Literal(0.92, datatype=XSD.double)))
g.add((stmt, EX.identifiedBy, Literal("AN/APG-83 NCTR")))

print(f"Source graph: {len(g)} triples")
print(f"  Blank nodes: {sum(1 for s in g.subjects() if isinstance(s, BNode))}")
print(f"  Typed subjects: {len(set(g.subjects(RDF.type, None)))}")

---
## 2. LPG Projection — The Full Collapse

`project_to_lpg()` does everything at once: types become node metadata,
literals become attributes, blank nodes inline, lists resolve.  The
result is a `ProjectedGraph` with nodes and edges — ready for
visualization, NetworkX, or JSON export.

In [ ]:
print(SEP)
print("  LPG PROJECTION — Full collapse")
print(SEP)

lpg = project_to_lpg(
    g,
    collapse_types=True,
    collapse_literals=True,
    resolve_blanks=True,
    resolve_lists=True,
)

print(f"\n{lpg}\n")

for iri, node in sorted(lpg.nodes.items()):
    label = node.label or iri.rsplit(":", 1)[-1]
    types_short = [t.rsplit(":", 1)[-1] for t in node.types]
    print(f"  Node: {label}")
    print(f"    types: {types_short}")
    for k, v in sorted(node.attributes.items()):
        k_short = k.rsplit(":", 1)[-1].rsplit("#", 1)[-1]
        print(f"    {k_short}: {v}")
    print()

print(f"  Edges:")
for edge in lpg.edges:
    s = edge.source.rsplit(":", 1)[-1]
    p = edge.predicate.rsplit(":", 1)[-1].rsplit("#", 1)[-1]
    t = edge.target.rsplit(":", 1)[-1]
    print(f"    {s} —{p}→ {t}")

**What happened:**
- `rdf:type` triples → `node.types` list (no type edges in output)
- `rdfs:label` → `node.label` (also in attributes)
- All literal-valued triples → `node.attributes` dict
- Blank node sensor suite → nested dict `{"radar": "AN/APG-83", ...}`
- RDF list loadout → Python list `["AIM-120D", "AIM-9X", "GBU-31"]`
- Object properties (`assigned_to`, `hasTarget`) → edges
- Reified statement node skipped (its subject is a named IRI, handled separately)

---
## 3. JSON Export

`ProjectedGraph.to_dict()` produces a JSON-serializable dict.

In [ ]:
import json

d = lpg.to_dict()
print(json.dumps(d, indent=2, default=str)[:2000])
print("...")

---
## 4. Reification Collapse

The identification statement is reified — it's a claim about a claim.
`collapse_reification()` unfolds it into a direct edge with metadata.

In [ ]:
print(SEP)
print("  REIFICATION COLLAPSE")
print(SEP)

reified_lpg = collapse_reification(g, preserve_metadata=True)
print(f"\n{reified_lpg}\n")

for edge in reified_lpg.edges:
    s = edge.source.rsplit(":", 1)[-1]
    p = edge.predicate.rsplit(":", 1)[-1]
    t = str(edge.target)[:40]
    print(f"  {s} —{p}→ {t}")
    for k, v in edge.attributes.items():
        k_short = k.rsplit(":", 1)[-1]
        print(f"    {k_short}: {v}")

The reified `rdf:Statement` became a direct edge `aircraft_001 —identifiedAs→ "F-16C Block 70"`
with `confidence: 0.92` and `identifiedBy: "AN/APG-83 NCTR"` as edge attributes.
This is exactly the LPG representation — properties on edges.

---
## 5. CONSTRUCT-Based Projections (Graph→Graph)

These stay in RDF.  Useful for intermediate processing, portal
generation, or storing simplified views in the holarchy.

In [ ]:
print(SEP)
print("  CONSTRUCT PROJECTIONS")
print(SEP)

# ── Strip types ──
q = build_construct(CONSTRUCT_STRIP_TYPES)
stripped = g.query(q).graph
type_count = len(list(stripped.triples((None, RDF.type, None))))
print(f"\n  STRIP_TYPES: {len(stripped)} triples, rdf:type count: {type_count}")

# ── Object properties only (topology) ──
q = build_construct(CONSTRUCT_OBJECT_PROPERTIES_ONLY)
topology = g.query(q).graph
print(f"  OBJECT_PROPERTIES_ONLY: {len(topology)} triples (pure topology)")
for s, p, o in sorted(topology):
    print(f"    {str(s).rsplit(':',1)[-1]:20s} "
          f"{str(p).rsplit(':',1)[-1]:15s} "
          f"{str(o).rsplit(':',1)[-1]}")

# ── Labels only ──
q = build_construct(CONSTRUCT_LABELS_ONLY)
labels = g.query(q).graph
print(f"\n  LABELS_ONLY: {len(labels)} triples")
for s, p, o in sorted(labels):
    print(f"    {str(s).rsplit(':',1)[-1]:20s} → {o}")

---
## 6. Projection Pipeline

Chain multiple steps — CONSTRUCT and/or Python transforms — into a
composable pipeline.  Fluent API.

In [ ]:
print(SEP)
print("  PROJECTION PIPELINE")
print(SEP)

pipeline = (
    ProjectionPipeline("viz-prep")
    .add_construct("strip_types", CONSTRUCT_STRIP_TYPES)
    .add_transform("strip_blanks", strip_blank_nodes)
    .add_transform("localize", localize_predicates)
)

print(f"\n  {pipeline}")
for i, step in enumerate(pipeline.steps):
    kind = "CONSTRUCT" if step.construct else "transform"
    print(f"    Step {i+1}: {step.name} ({kind})")

# Apply to get RDF
result_graph = pipeline.apply(g)
print(f"\n  Pipeline output: {len(result_graph)} triples")
print(f"  Sample predicates (localized):")
preds = sorted(set(str(p) for _, p, _ in result_graph))[:8]
for p in preds:
    print(f"    {p}")

# Apply to get LPG directly
lpg_from_pipeline = pipeline.apply_to_lpg(g)
print(f"\n  Pipeline → LPG: {lpg_from_pipeline}")

---
## 7. Utility Functions

Building blocks for custom projections.

In [ ]:
print(SEP)
print("  UTILITY FUNCTIONS")
print(SEP)

# ── Extract types (companion to type-stripping) ──
types = extract_types(g)
print("\n  extract_types:")
for iri, type_list in sorted(types.items()):
    name = iri.rsplit(":", 1)[-1]
    ts = [t.rsplit(":", 1)[-1] for t in type_list]
    print(f"    {name:25s} → {ts}")

# ── Filter by class ──
aircraft_only = filter_by_class(g, str(EX.Aircraft))
print(f"\n  filter_by_class(Aircraft): {len(aircraft_only)} triples")
subjects = sorted(set(str(s).rsplit(":", 1)[-1] for s in aircraft_only.subjects()))
print(f"    Subjects: {subjects}")

# ── Merge graphs ──
g1 = Graph()
g1.add((EX.a, EX.p, Literal("from g1")))
g2 = Graph()
g2.add((EX.b, EX.q, Literal("from g2")))
merged = g1 + g2
print(f"\n  merge_graphs: {len(g1)} + {len(g2)} = {len(merged)} triples")

---
## 8. Holarchy-Level Projections

Project individual holons or the entire holarchy topology.  Results
can be stored as projection named graphs in the dataset.

In [ ]:
print(SEP)
print("  HOLARCHY-LEVEL PROJECTIONS")
print(SEP)

ds = HolonicDataset()

# Build a small holarchy
ds.add_holon("urn:holon:ops", "Operations")
ds.add_holon("urn:holon:air", "Air Picture",
             member_of="urn:holon:ops")
ds.add_holon("urn:holon:ground", "Ground Picture",
             member_of="urn:holon:ops")

# Add aircraft data to Air Picture
ds.add_interior("urn:holon:air", g.serialize(format="turtle"))

# Add a portal
ds.add_holon("urn:holon:display", "COP Display",
             member_of="urn:holon:ops")
ds.add_portal(
    "urn:portal:air-to-display",
    "urn:holon:air",
    "urn:holon:display",
    label="Air → Display",
    construct_query=build_construct(CONSTRUCT_OBJECT_PROPERTIES_ONLY,
                                    "urn:holon:air/interior"),
)

# ── Project a single holon to LPG ──
air_lpg = ds.project_holon("urn:holon:air")
print(f"\n  project_holon('Air Picture'): {air_lpg}")
print(f"    Nodes: {len(air_lpg.nodes)}")
print(f"    Edges: {len(air_lpg.edges)}")

# Store the projection back into the holarchy
air_lpg_stored = ds.project_holon(
    "urn:holon:air",
    store_as="urn:holon:air/projection/viz",
    collapse_types=True,
    resolve_blanks=True,
)
print(f"    Stored as named graph: urn:holon:air/projection/viz")

# ── Project the holarchy topology ──
topo = ds.project_holarchy()
print(f"\n  project_holarchy(): {topo}")
print(f"    Nodes (holons + portals):")
for iri, node in sorted(topo.nodes.items()):
    label = node.label or iri.rsplit(":", 1)[-1]
    types_short = [t.rsplit(":", 1)[-1] for t in node.types]
    print(f"      {label:25s}  {types_short}")
print(f"    Edges (membership + portals):")
for edge in topo.edges:
    s = edge.source.rsplit(":", 1)[-1]
    p = edge.predicate.rsplit(":", 1)[-1].rsplit("#", 1)[-1]
    t = edge.target.rsplit(":", 1)[-1]
    print(f"      {s} —{p}→ {t}")

---
## 9. Pipeline Applied to a Holon

Apply a full pipeline to a holon's merged interiors and optionally
store the result back as a projection named graph.

In [ ]:
print(CONSTRUCT_LABELS_ONLY)

In [ ]:
from holonic.projections import ProjectionStep

In [ ]:
print(SEP)
print("  PIPELINE APPLIED TO HOLON")
print(SEP)

a = """
CONSTRUCT {
    ?s ?p ?o .
}
WHERE {
    GRAPH ?g { ?s ?p ?o . FILTER(isIRI(?o)) }
}
"""

b = """
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX skos: <http://www.w3.org/2004/02/skos/core#>
CONSTRUCT {
    ?s rdfs:label ?label .
}
WHERE {
    graph ?g {
        { ?s rdfs:label ?label }
        UNION
        { ?s skos:prefLabel ?label }
        UNION
        { ?s skos:altLabel ?label }
    }
}
"""

viz_pipeline = (
    ProjectionPipeline("air-viz")
    #.add_construct("topology", CONSTRUCT_OBJECT_PROPERTIES_ONLY)
    #.add_construct("labels", CONSTRUCT_LABELS_ONLY)
)
viz_pipeline.steps.extend([
    ProjectionStep(name='a', construct=a),
    ProjectionStep(name='b', construct=b)
])

# Note: these CONSTRUCTs run against the holon's merged interior,
# not the full dataset — so no GRAPH clause needed.
result = ds.apply_pipeline(
    "urn:holon:air",
    viz_pipeline,
    store_as="urn:holon:air/projection/topology",
)

print(f"\n  Pipeline '{viz_pipeline.name}' applied to Air Picture")
print(f"  Result: {len(result)} triples")
print(f"  Stored as: urn:holon:air/projection/topology")

for s, p, o in sorted(result):
    print(f"    {str(s).rsplit(':',1)[-1]:20s} "
          f"{str(p).rsplit(':',1)[-1].rsplit('#',1)[-1]:15s} "
          f"{str(o)[:40]}")

---
## Summary

| Projection | Mode | Input | Output | Use case |
|---|---|---|---|---|
| `project_to_lpg()` | Pythonic | Graph | ProjectedGraph | Visualization, JSON API, NetworkX |
| `collapse_reification()` | Pythonic | Graph | ProjectedGraph | Edge-attributed graphs |
| `CONSTRUCT_STRIP_TYPES` | CONSTRUCT | Graph | Graph | Remove type clutter |
| `CONSTRUCT_OBJECT_PROPERTIES_ONLY` | CONSTRUCT | Graph | Graph | Extract topology |
| `CONSTRUCT_LABELS_ONLY` | CONSTRUCT | Graph | Graph | Label overlay |
| `CONSTRUCT_SUBCLASS_TREE` | CONSTRUCT | Graph | Graph | Class hierarchy |
| `ProjectionPipeline` | Composable | Graph | Graph or LPG | Multi-step transforms |
| `ds.project_holon()` | Client | Holon | ProjectedGraph | Single holon → LPG |
| `ds.project_holarchy()` | Client | Dataset | ProjectedGraph | Topology visualization |
| `ds.apply_pipeline()` | Client | Holon | Graph | Pipeline on holon interiors |

The key design principle: **CONSTRUCT stays in RDF** (storable as named graphs,
traversable by portals, queryable via SPARQL).  **Pythonic projections exit RDF**
(JSON-serializable, consumable by visualization tools, compatible with LPG
ecosystems).  Pipelines compose both.  The holarchy stores projection results
as first-class named graphs via `cga:hasProjection`.